# Loan Prediction Competition

In this notebook, we apply ensemble methods such as XGBoost for binary loan approval classification, featuring systematic hyperparameter optimization and SHAP model interpretability.


## 1. Setup & Configuration


In [ ]:
import os
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Configuration constants — edit DATA_DIR to point to a different directory
DATA_DIR = '.'
RANDOM_STATE = 42
CV_FOLDS = 5


**Dependencies:** pandas, numpy, scikit-learn, xgboost, shap, matplotlib, seaborn

Install with: `pip install -r requirements.txt`


## 2. Data Loading & Exploration

Train and test data are loaded once here and reused in later sections — no duplicate loads.


In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_no_label.csv'))
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()


In [ ]:
train_df.describe()


In [ ]:
# =====================================
# Exploración de distribuciones
# =====================================
import matplotlib.pyplot as plt
import seaborn as sns

# Definimos listas de variables
categoricas = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
               'Credit_History', 'Property_Area', 'Loan_Status']  # target incluida para ver balance
numericas = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

# -------------------------
# Categóricas: conteo
# -------------------------
for col in categoricas:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=train_df, palette="pastel")
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Conteo")
    plt.xticks(rotation=45)
    plt.show()

# -------------------------
# Numéricas: histograma + boxplot
# -------------------------
for col in numericas:
    fig, axes = plt.subplots(1,2, figsize=(12,4))

    # Histograma
    sns.histplot(train_df[col].dropna(), kde=True, ax=axes[0], color="skyblue")
    axes[0].set_title(f"Histograma de {col}")

    # Boxplot para ver outliers
    sns.boxplot(x=train_df[col], ax=axes[1], color="lightgreen")
    axes[1].set_title(f"Boxplot de {col}")

    plt.tight_layout()
    plt.show()

In [ ]:
# Revisión de valores nulos
missing_values = train_df.isnull().sum()
missing_percent = (missing_values / len(train_df)) * 100

missing_summary = pd.DataFrame({
    'Valores Nulos': missing_values,
    'Porcentaje (%)': missing_percent
}).sort_values(by='Porcentaje (%)', ascending=False)

print("Resumen de valores nulos:")
display(missing_summary)